# Text Preprocessing
In NLP, text preprocessing is a very important and also first step that **transforms raw/ unstructured data into clean standardized data**.



In [106]:
import numpy as np
import pandas as pd
import string
import re

In [46]:
full_df = pd.read_csv(
    "data/Customer-Support-on-Twitter-twcs.csv",
    nrows=5000
)

full_df.head(2)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0


In [54]:
df = full_df[["text"]]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    5000 non-null   str  
dtypes: str(1)
memory usage: 573.7 KB


## 1. Lower Casing

In [68]:
df["text"] = df["text"].astype('str')

def lower_case(text):
    text = text.str.lower()
    return text

df["text_lower"] = df["text"].str.lower()
df.head()

,text,text_lower
0,@115712 I understand. I would like to assist y...,@115712 i understand. i would like to assist y...
1,@sprintcare and how do you propose we do that,@sprintcare and how do you propose we do that
2,@sprintcare I have sent several private messag...,@sprintcare i have sent several private messag...
3,@115712 Please send us a Private Message so th...,@115712 please send us a private message so th...
4,@sprintcare I did.,@sprintcare i did.


## 2. Removal of Punctuations

in puthon contains the following punctuations symbols  
**`string.punctuations`** -> !"#$%&\'()*+,-./:;<=>?@[\\]^_{|}~`  
we choose the list of punctuation to exclude depending on the use case. 

In [69]:
df.drop(["text_lower"], axis=1, inplace=True)

In [70]:
PUNCT_TO_REMOVE = string.punctuation
print(f"Punctuation which going to remove: {PUNCT_TO_REMOVE}")

def remove_punctuation(text):
    """custom function to remove the Punctuation"""
    text = text.translate(str.maketrans('', '', PUNCT_TO_REMOVE)) 
    # p1 and p2 (replacment work) => trun 'a' into 'b'
    # p3 =>  ch which completely delete.
    return text

df["text_without_punct"] = df["text"].apply(remove_punctuation)
df.head()

Punctuation which going to remove: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


,text,text_without_punct
0,@115712 I understand. I would like to assist y...,115712 I understand I would like to assist you...
1,@sprintcare and how do you propose we do that,sprintcare and how do you propose we do that
2,@sprintcare I have sent several private messag...,sprintcare I have sent several private message...
3,@115712 Please send us a Private Message so th...,115712 Please send us a Private Message so tha...
4,@sprintcare I did.,sprintcare I did


## 3. Removal of Stopwords
commonly occuring words in a language like: 'the', 'a', on and more...  
because they don't provide valuable information for downstream analysis.

We use: **NLTK** - Natural Language Tool Kit

It ia a popular python labrary  
which is used for Netural language(NLP) operation.

In [129]:
# nltk.download("punkt")
# nltk.download("punkt_tab")
# nltk.download("stopwords")

In [75]:
import nltk
from nltk.corpus import stopwords

print("The Stopword list for English languages: ")
", ".join(stopwords.words('english'))

The Stopword list for English languages: 


"a, about, above, after, again, against, ain, all, am, an, and, any, are, aren, aren't, as, at, be, because, been, before, being, below, between, both, but, by, can, couldn, couldn't, d, did, didn, didn't, do, does, doesn, doesn't, doing, don, don't, down, during, each, few, for, from, further, had, hadn, hadn't, has, hasn, hasn't, have, haven, haven't, having, he, he'd, he'll, her, here, hers, herself, he's, him, himself, his, how, i, i'd, if, i'll, i'm, in, into, is, isn, isn't, it, it'd, it'll, it's, its, itself, i've, just, ll, m, ma, me, mightn, mightn't, more, most, mustn, mustn't, my, myself, needn, needn't, no, nor, not, now, o, of, off, on, once, only, or, other, our, ours, ourselves, out, over, own, re, s, same, shan, shan't, she, she'd, she'll, she's, should, shouldn, shouldn't, should've, so, some, such, t, than, that, that'll, the, their, theirs, them, themselves, then, there, these, they, they'd, they'll, they're, they've, this, those, through, to, too, under, until, up, 

In [81]:
STOPWORDS = set(stopwords.words('english'))

def remove_stopwords(text):
    text = " ".join([word for word in str(text).split() if word not in STOPWORDS])
    return text

df["text_without_stop"] = df["text_without_punct"].apply(remove_stopwords)
df.head()

,text,text_without_punct,text_without_stop
0,@115712 I understand. I would like to assist y...,115712 I understand I would like to assist you...,115712 I understand I would like assist We wou...
1,@sprintcare and how do you propose we do that,sprintcare and how do you propose we do that,sprintcare propose
2,@sprintcare I have sent several private messag...,sprintcare I have sent several private message...,sprintcare I sent several private messages one...
3,@115712 Please send us a Private Message so th...,115712 Please send us a Private Message so tha...,115712 Please send us Private Message assist J...
4,@sprintcare I did.,sprintcare I did,sprintcare I


## 4. Removal of Frequent words
previous step: we removed the stopwords based on a language info.

but now: if we have a domain specific corpus, we might also have some frequent words which are of not so much importance to us.

here we remove: **frequents words in the given corpus**
>Note: **TFIDF** automatically taken care this 

In [83]:
from collections import Counter
cnt = Counter()

for text in df["text_without_stop"].values:
    for word in text.split():
        cnt[word] += 1

cnt.most_common(10)

[('I', 1437),
 ('us', 752),
 ('DM', 514),
 ('help', 479),
 ('Please', 376),
 ('We', 338),
 ('Hi', 293),
 ('Thanks', 287),
 ('get', 279),
 ('please', 247)]

In [87]:
FREQWORDS = set([w for (w, wc) in cnt.most_common(10)])

def remove_freqwords(text):
    text = " ".join([word for word in str(text).split() if word not in FREQWORDS])
    return text

df["text_wo_stopfreq"] = df["text_without_stop"].apply(remove_freqwords)
df.head()

,text,text_without_punct,text_without_stop,text_wo_stopfreq
0,@115712 I understand. I would like to assist y...,115712 I understand I would like to assist you...,115712 I understand I would like assist We wou...,115712 understand would like assist would need...
1,@sprintcare and how do you propose we do that,sprintcare and how do you propose we do that,sprintcare propose,sprintcare propose
2,@sprintcare I have sent several private messag...,sprintcare I have sent several private message...,sprintcare I sent several private messages one...,sprintcare sent several private messages one r...
3,@115712 Please send us a Private Message so th...,115712 Please send us a Private Message so tha...,115712 Please send us Private Message assist J...,115712 send Private Message assist Just click ...
4,@sprintcare I did.,sprintcare I did,sprintcare I,sprintcare


In [88]:
df.drop(["text_without_punct", "text_without_stop"], axis=1, inplace=True)

In [89]:
n_rare_words = 10
RAREWORDS = set([w for (w, wc) in cnt.most_common()[: -n_rare_words-1: -1]])

def remove_rarewords(text):
    text = " ".join([word for word in str(text).split() if word not in RAREWORDS])
    return text

df["text_wo_stopfreqrare"] = df["text_wo_stopfreq"].apply(remove_rarewords)
df.head()

,text,text_wo_stopfreq,text_wo_stopfreqrare
0,@115712 I understand. I would like to assist y...,115712 understand would like assist would need...,115712 understand would like assist would need...
1,@sprintcare and how do you propose we do that,sprintcare propose,sprintcare propose
2,@sprintcare I have sent several private messag...,sprintcare sent several private messages one r...,sprintcare sent several private messages one r...
3,@115712 Please send us a Private Message so th...,115712 send Private Message assist Just click ...,115712 send Private Message assist Just click ...
4,@sprintcare I did.,sprintcare,sprintcare


We can combine all the list of words (stopwords, frequent words and rare words) and create a single list to remove them at once.

In [92]:
df.drop(["text_wo_stopfreq", "text_wo_stopfreqrare"], axis=1, inplace=True)

## 5. Stremming
Stemming is the process of reducing inflected (or sometimes derived) words to their word stem, base or root form

walks/walking => walk

In [94]:
from nltk.stem.porter import PorterStemmer # this one is famous , waidly used

stemmer = PorterStemmer()

def stem_words(text):
    text = " ".join([stemmer.stem(word) for word in str(text).split()])
    return text

df["text_stemmed"] = df["text"].apply(stem_words)
df.head()

,text,text_stemmed
0,@115712 I understand. I would like to assist y...,@115712 i understand. i would like to assist y...
1,@sprintcare and how do you propose we do that,@sprintcar and how do you propos we do that
2,@sprintcare I have sent several private messag...,@sprintcar i have sent sever privat messag and...
3,@115712 Please send us a Private Message so th...,@115712 pleas send us a privat messag so that ...
4,@sprintcare I did.,@sprintcar i did.


here `private` and `propose` have thier `e` at the end chopped off due to stemming.
SO for that we use -> **Lemmatization**

### Lemmatization
* it is similar to stemming but it makes sure the root word (lemma) belongs to the language.
* it is generally **slower** than stremming process

In [130]:
# nltk.download('wordnet')
# nltk.download('averaged_perceptron_tagger_eng')

In [99]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_words(text):
    text = " ".join([lemmatizer.lemmatize(word) for word in str(text).split()])
    return text

df["text_lemmatized"] = df["text"].apply(lemmatize_words)
df.head()

,text,text_stemmed,text_lemmatized
0,@115712 I understand. I would like to assist y...,@115712 i understand. i would like to assist y...,@115712 I understand. I would like to assist y...
1,@sprintcare and how do you propose we do that,@sprintcar and how do you propos we do that,@sprintcare and how do you propose we do that
2,@sprintcare I have sent several private messag...,@sprintcar i have sent sever privat messag and...,@sprintcare I have sent several private messag...
3,@115712 Please send us a Private Message so th...,@115712 pleas send us a privat messag so that ...,@115712 Please send u a Private Message so tha...
4,@sprintcare I did.,@sprintcar i did.,@sprintcare I did.


>Note: Lemmatization process depends on the POS tag to come up with the correct lemma.

In [100]:
print("Word is : stripes")
print("Lemma result for verb : ",lemmatizer.lemmatize("stripes", 'v'))
print("Lemma result for noun : ",lemmatizer.lemmatize("stripes", 'n'))

Word is : stripes
Lemma result for verb :  strip
Lemma result for noun :  stripe


so now let us redo the lemmatization process for our dataset

In [105]:
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
wordnet_map = {"N": wordnet.NOUN, "V": wordnet.VERB, "J": wordnet.ADJ, "R": wordnet.ADV}

def lemmatize_words2(text):
    pos_tagged_text = nltk.pos_tag(text.split()) # parts of speech tag text
    text = " ".join([lemmatizer.lemmatize(word, wordnet_map.get(pos[0], wordnet.NOUN)) for word, pos in pos_tagged_text])
    return text

df["text_lemmatized"] = df["text"].apply(lemmatize_words2)
df.head()

,text,text_stemmed,text_lemmatized
0,@115712 I understand. I would like to assist y...,@115712 i understand. i would like to assist y...,@115712 I understand. I would like to assist y...
1,@sprintcare and how do you propose we do that,@sprintcar and how do you propos we do that,@sprintcare and how do you propose we do that
2,@sprintcare I have sent several private messag...,@sprintcar i have sent sever privat messag and...,@sprintcare I have send several private messag...
3,@115712 Please send us a Private Message so th...,@115712 pleas send us a privat messag so that ...,@115712 Please send u a Private Message so tha...
4,@sprintcare I did.,@sprintcar i did.,@sprintcare I did.


## 6. Removal of URL

We use: **re-Regual expression**

inside it we use - **re.sub(pattern, repl, string, count=0, flags=0)**  

pattern:  
\S - non-white space character  
\s - white space character


In [109]:
def remove_urls(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    return text

In [110]:
text = "Driverless AI NLP blog post on https://www.h2o.ai/blog/detecting-sarcasm-is-difficult-but-ai-may-have-an-answer/"
remove_urls(text)

'Driverless AI NLP blog post on '

In [111]:
text = "Please refer to link http://lnkd.in/ecnt5yC for the paper"
remove_urls(text)

'Please refer to link  for the paper'

In [112]:
text = "Want to know more. Checkout www.h2o.ai for additional information"
remove_urls(text)

'Want to know more. Checkout  for additional information'

## 7. Removal of HTML Tags
- **`.`** => any charcter exist
- **`*`** => 0 or more charcter
- **`.*`** => jitne bhi words ho
- ***`?`*** => remove in non greedy manaers
- Non-greedy maners: pattern identify till the **`>`**

In [121]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

In [124]:
text = """
<div>
    <h1>Customer Support Notice</h1>
    <p>Please click <a href="https://twitter.com">here</a> to reset your password.</p>
    <br>
    <p>Ensure your new password is <b>strong</b> and <i>unique</i>.</p>
</div>
"""

print(remove_html(text))



    Customer Support Notice
    Please click here to reset your password.
    
    Ensure your new password is strong and unique.




we can also use `BeautifulSoup` to get the text from HTML document

In [128]:
from bs4 import BeautifulSoup

def remove_html(text):
    return BeautifulSoup(text, "html.parser").text

print(remove_html(text))



Customer Support Notice
Please click here to reset your password.

Ensure your new password is strong and unique.


